# CQL2 Search, Collection Search & Geoid Lookup

**Personas**
- **GS — Geospatial Scientist**: queries items using spatial/temporal/attribute predicates, paginates large result sets.
- **SI — System Integrator**: wires collection discovery and geoid-based asset resolution to downstream tools (e.g. TiTiler tile server).

## Structure

| Part | Topic |
|------|-------|
| 1 | CQL2 Spatial + Temporal + Attribute Search with pagination |
| 2 | Full-text Collection Search across catalogs |
| 3 | Geoid-based Lookup for TiTiler integration |

## Prerequisites

- A running GeoID / DynaStore instance.
- A `.env` file (or exported shell variables) containing:
  - `DYNASTORE_BASE_URL` — e.g. `http://localhost:8080`
  - `DYNASTORE_TOKEN` — bearer token
  - `CATALOG_ID` — target catalog identifier
  - `COLLECTION_ID` — target collection identifier
- `pip install httpx python-dotenv`

In [ ]:
import os
from dotenv import load_dotenv
import httpx

load_dotenv()

BASE_URL    = os.environ["DYNASTORE_BASE_URL"].rstrip("/")
TOKEN       = os.environ["DYNASTORE_TOKEN"]
CATALOG_ID  = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)

print(f"Base URL    : {BASE_URL}")
print(f"Catalog     : {CATALOG_ID}")
print(f"Collection  : {COLLECTION_ID}")

## Part 1 — CQL2 Spatial + Temporal + Attribute Search

GeoID exposes a STAC-compliant `/stac/catalogs/{catalog_id}/search` endpoint that accepts both **CQL2-JSON** (POST body `filter` key) and **CQL2-Text** (GET query parameter `filter`).  
Filters are evaluated by the pygeofilter pipeline before reaching the storage driver.

The region of interest below covers the **Horn of Africa** — a primary FAO crop-monitoring domain for dekadal NDVI and rainfall products.

In [ ]:
# --- CQL2-JSON POST search ---
search_body = {
    "bbox": [33.0, 3.0, 51.5, 22.0],          # Horn of Africa
    "datetime": "2024-01-01T00:00:00Z/2024-03-31T23:59:59Z",
    "filter": {
        "op": ">=",
        "args": [{"property": "eo:cloud_cover"}, 0]
    },
    "filter-lang": "cql2-json",
    "limit": 10,
    "sortby": ["-datetime"]
}

resp = client.post(f"/stac/catalogs/{CATALOG_ID}/search", json=search_body)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
assert "features" in data, "Response must contain a 'features' key"

returned = data.get("context", {}).get("returned", len(data["features"]))
print(f"Returned    : {returned} items")
print(f"Total matched (context): {data.get('context', {}).get('matched', 'n/a')}")
for feat in data["features"][:3]:
    props = feat.get("properties", {})
    print(f"  id={feat['id']}  datetime={props.get('datetime', '?')}  cloud={props.get('eo:cloud_cover', '?')}")

In [ ]:
# --- CQL2-Text GET search ---
# The same predicate expressed as CQL2-Text
cql2_text_filter = 'eo:cloud_cover >= 0'

params = {
    "bbox": "33.0,3.0,51.5,22.0",
    "datetime": "2024-01-01T00:00:00Z/2024-03-31T23:59:59Z",
    "filter": cql2_text_filter,
    "filter-lang": "cql2-text",
    "limit": 10,
    "sortby": "-datetime"
}

resp_text = client.get(f"/stac/catalogs/{CATALOG_ID}/search", params=params)
assert resp_text.status_code == 200, f"CQL2-Text search failed: {resp_text.status_code}: {resp_text.text}"

data_text = resp_text.json()
assert "features" in data_text, "Response must contain a 'features' key"

returned_text = data_text.get("context", {}).get("returned", len(data_text["features"]))
print(f"CQL2-Text returned: {returned_text} items")

# Both dialects must agree on item IDs for the first page
ids_json = {f["id"] for f in data["features"]}
ids_text = {f["id"] for f in data_text["features"]}
assert ids_json == ids_text, (
    f"CQL2-JSON and CQL2-Text results differ.\n  JSON: {ids_json}\n  Text: {ids_text}"
)
print("CQL2-JSON and CQL2-Text results match.")

In [ ]:
# --- Pagination: follow next link ---
links = data.get("links", [])
next_link = next((lnk for lnk in links if lnk.get("rel") == "next"), None)

if next_link is None:
    print("No next page — result set fits in one page, skipping pagination check.")
else:
    # next href may be absolute or relative
    href = next_link["href"]
    if href.startswith("http"):
        resp_p2 = httpx.get(href, headers=headers, timeout=30.0)
    else:
        resp_p2 = client.get(href)

    assert resp_p2.status_code == 200, f"Page 2 request failed: {resp_p2.status_code}: {resp_p2.text}"
    data_p2 = resp_p2.json()
    assert "features" in data_p2, "Page 2 response must contain a 'features' key"

    ids_p1 = {f["id"] for f in data["features"]}
    ids_p2 = {f["id"] for f in data_p2["features"]}

    assert ids_p1.isdisjoint(ids_p2), (
        f"Page 1 and Page 2 share item IDs — pagination is broken.\nOverlap: {ids_p1 & ids_p2}"
    )
    print(f"Page 1 items : {len(ids_p1)}")
    print(f"Page 2 items : {len(ids_p2)}")
    print("Pages are disjoint — pagination works correctly.")

## Part 2 — Full-text Collection Search

GeoID provides two collection search endpoints:

| Endpoint | Scope |
|----------|-------|
| `POST /stac/collections-search` | All catalogs the user can read |
| `GET /search/catalogs/{catalog_id}/collections-search` | Single catalog |

Both accept a `q` query parameter for free-text matching against collection title/description.

In [ ]:
# --- Global collection search: q=rainfall ---
resp_cs = client.post("/stac/collections-search", json={"q": "rainfall"})
assert resp_cs.status_code == 200, f"Global collection search failed: {resp_cs.status_code}: {resp_cs.text}"

cs_data = resp_cs.json()
collections = cs_data.get("collections", cs_data if isinstance(cs_data, list) else [])
print(f"Found {len(collections)} rainfall collection(s):")
for col in collections:
    title = col.get("title") or col.get("id", "?")
    print(f"  [{col.get('id', '?')}] {title}")

In [ ]:
# --- Catalog-scoped collection search: q=ndvi ---
resp_ndvi = client.get(
    f"/search/catalogs/{CATALOG_ID}/collections-search",
    params={"q": "ndvi"}
)
assert resp_ndvi.status_code == 200, f"Catalog-scoped search failed: {resp_ndvi.status_code}: {resp_ndvi.text}"

ndvi_data = resp_ndvi.json()
ndvi_cols = ndvi_data.get("collections", ndvi_data if isinstance(ndvi_data, list) else [])

# All returned collections must belong to the requested catalog
for col in ndvi_cols:
    links_col = col.get("links", [])
    root_href = next(
        (lnk["href"] for lnk in links_col if lnk.get("rel") == "root"), ""
    )
    # If link data is available, verify catalog scoping
    if root_href:
        assert CATALOG_ID in root_href, (
            f"Collection {col.get('id')} root link {root_href!r} does not reference catalog {CATALOG_ID}"
        )

print(f"Found {len(ndvi_cols)} NDVI collection(s) in catalog '{CATALOG_ID}':")
for col in ndvi_cols:
    print(f"  [{col.get('id', '?')}] {col.get('title', '?')}")

In [ ]:
# --- Empty q → returns all accessible collections ---
resp_all = client.post("/stac/collections-search", json={"q": ""})
assert resp_all.status_code == 200, f"Empty-q search failed: {resp_all.status_code}: {resp_all.text}"

all_data = resp_all.json()
all_cols = all_data.get("collections", all_data if isinstance(all_data, list) else [])

assert len(all_cols) > 0, "Empty-q search must return at least one collection"
print(f"Empty query returned {len(all_cols)} collection(s) — all accessible collections.")

## Part 3 — Geoid-based Lookup (TiTiler integration)

A **geoid** is GeoID's compact spatial identifier for a geographic cell (geohash-derived).  
The lookup endpoint resolves which collections cover a given geoid cell — enabling TiTiler to discover tile-able assets without a full STAC search.

Endpoint: `GET /search/catalogs/{catalog_id}/geoid/{geoid}`

In [ ]:
# --- Happy path: geoid lookup ---
# A geohash covering Addis Ababa (~9°N 38°E)
GEOID = os.environ.get("GEOID", "s174")

resp_geo = client.get(f"/search/catalogs/{CATALOG_ID}/geoid/{GEOID}")
assert resp_geo.status_code == 200, (
    f"Geoid lookup failed: {resp_geo.status_code}: {resp_geo.text}"
)

geo_data = resp_geo.json()
print(f"Geoid '{GEOID}' found in collections:")
for col in geo_data.get("collections", []):
    print(f"  {col}")

In [ ]:
# --- Error cases ---

# 1. Missing catalog_id in query string → 400
resp_no_cat = client.get(f"/search/geoid/{GEOID}")
assert resp_no_cat.status_code in (400, 404, 422), (
    f"Expected 4xx without catalog_id, got {resp_no_cat.status_code}"
)
print(f"Missing catalog → HTTP {resp_no_cat.status_code} (expected 4xx)")

# 2. Providing both geoids and external_id must be rejected
resp_conflict = client.get(
    f"/search/catalogs/{CATALOG_ID}/geoid/{GEOID}",
    params={"external_id": "some-external-id"}
)
assert resp_conflict.status_code in (400, 422), (
    f"Expected 400/422 for geoid+external_id conflict, got {resp_conflict.status_code}: {resp_conflict.text}"
)
print(f"Geoid + external_id conflict → HTTP {resp_conflict.status_code} (expected 400/422)")

## Driver Capability Check

Some storage drivers do not implement `Capability.SEARCH`.  
When the driver does not expose search, GeoID returns **HTTP 501 Not Implemented** rather than silently returning empty results.

In [ ]:
resp_drv = client.get("/configs/storage/drivers")
assert resp_drv.status_code == 200, f"Driver list failed: {resp_drv.status_code}: {resp_drv.text}"

drivers = resp_drv.json()
if isinstance(drivers, dict):
    drivers = list(drivers.values())

print(f"Registered storage drivers ({len(drivers)}):")
search_capable = []
for drv in drivers:
    caps = drv.get("capabilities", [])
    has_search = "SEARCH" in [c.upper() for c in caps]
    marker = "[SEARCH]" if has_search else "[no search]"
    print(f"  {drv.get('driver_id', drv.get('id', '?'))} {marker} — caps: {caps}")
    if has_search:
        search_capable.append(drv.get("driver_id", drv.get("id", "?")))

if search_capable:
    print(f"\nDrivers with SEARCH capability: {search_capable}")
else:
    print("\nNo driver advertises SEARCH — all search endpoints will return 501.")
    print("Tip: configure a driver that implements Capability.SEARCH (e.g. PostgreSQL, Elasticsearch).")